
# Positive Case Report Prompt Lab

This notebook is for iterating on **report-generation prompts** for the validation positive-case task.

The intended workflow is narrow on purpose:
1. Pick the memo source mode (`both`, `data_only`, or `paper_only`).
2. Edit the report-generation prompt addendum.
3. Generate a **named report variant**.
4. Evaluate prediction performance by referring to that variant name.

Fixed choices in this notebook:
- report model: `gpt-4.1`
- prediction model: `gpt-4.1`
- prediction elicitation: reasoning JSON
- prediction mode: one question at a time
- report augmentation wrapper: fixed
- prediction calls are made **sequentially**, so progress appears question by question


In [ ]:

from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "positive_cases").exists() and (p / "input").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from positive_cases import notebook_utils as pc_utils



## API key

Set your API key in the terminal before launching Jupyter:

```bash
export OPENAI_API_KEY="your_key_here"
```

This notebook reads the standard OpenAI environment variable. It does not store the key in the notebook.


In [ ]:

REPORT_MODEL = "gpt-4.1"
PREDICTION_MODEL = "gpt-4.1"
TEMPERATURE = 1.0
MAX_TOKENS = None
PAUSE_SECONDS = 0.0
N_BOOT = 2000

RUN_REPORT_GENERATION = False
OVERWRITE_REPORT_VARIANT = False
RUN_PREDICTIONS = False
RUN_NAME = "gpt41_report_prompt_lab"

SOURCE_MODE = "both"
BASE_REPORT_STYLE = "structured"
NEW_VARIANT_NAME = "both_structured_custom_v1"

PREDICTION_VARIANTS = [
    "baseline_reasoning",
    "both_structured",
    # Add NEW_VARIANT_NAME here after you generate it.
]

if RUN_REPORT_GENERATION or RUN_PREDICTIONS:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set in the environment."
else:
    print("OPENAI_API_KEY is only required when RUN_REPORT_GENERATION or RUN_PREDICTIONS is True.")


In [ ]:

validation_df = pc_utils.load_validation_df(REPO_ROOT)
variant_registry = pc_utils.load_report_variant_registry(REPO_ROOT, refresh=True)
variant_df = pc_utils.report_variant_registry_dataframe(variant_registry)

display(
    variant_df[
        [
            "variant_name",
            "variant_type",
            "source_mode",
            "report_style",
            "report_path",
            "report_prompt_path",
        ]
    ].sort_values(["variant_type", "variant_name"]).reset_index(drop=True)
)
validation_df.head()



## Where existing variants live

Every report variant is recorded in:
- `positive_cases/output/report_variant_registry.json`

For each variant, the registry stores paths for:
- `agentic_report.md`
- `report_generation_prompt.md`

Built-in variants have now also been backfilled with prompt records, so collaborators can inspect what has already been tried before making a new one.


In [ ]:

SELECTED_EXISTING_VARIANT = "both_structured"
selected_entry = variant_registry[SELECTED_EXISTING_VARIANT]
selected_report_path = REPO_ROOT / selected_entry["report_path"]
selected_prompt_path = REPO_ROOT / selected_entry["report_prompt_path"]

print(f"Selected variant: {SELECTED_EXISTING_VARIANT}")
print(f"Report path: {selected_report_path}")
print(f"Prompt path: {selected_prompt_path}")
print(f"Variant type: {selected_entry['variant_type']}")
print(f"Source mode: {selected_entry['source_mode']}")
print(f"Report style: {selected_entry['report_style']}")


In [ ]:

selected_prompt_text = selected_prompt_path.read_text(encoding="utf-8")
selected_report_text = selected_report_path.read_text(encoding="utf-8")

print("Existing report-generation prompt record:")
preview_chars = 4000
print(selected_prompt_text[:preview_chars])
if len(selected_prompt_text) > preview_chars:
    print()
    print("... [prompt record truncated for display] ...")


In [ ]:

print("Existing generated report:")
preview_chars = 4000
print(selected_report_text[:preview_chars])
if len(selected_report_text) > preview_chars:
    print()
    print("... [report truncated for display] ...")



## Editable part: report-generation prompt

This is the part to revise.

The notebook builds the full report-generation prompt from:
- the chosen `SOURCE_MODE`
- the chosen `BASE_REPORT_STYLE`
- the existing source memos already on disk
- the addendum below

The intended thing to edit is `REPORT_PROMPT_ADDENDUM`.


In [ ]:

REPORT_PROMPT_ADDENDUM = """Optional extra instructions for this custom report variant.

Edit this block to try prompt changes.

Examples:
- Emphasize moderator interactions over global averages.
- Prefer compact operational rules over narrative discussion.
- Be conservative about claims not strongly grounded in the memos.
"""

REPORT_GENERATION_PROMPT = pc_utils.build_report_generation_prompt(
    source_mode=SOURCE_MODE,
    report_style=BASE_REPORT_STYLE,
    prompt_addendum=REPORT_PROMPT_ADDENDUM,
    repo_root=REPO_ROOT,
)

preview_chars = 7000
print(REPORT_GENERATION_PROMPT[:preview_chars])
if len(REPORT_GENERATION_PROMPT) > preview_chars:
    print()
    print("... [prompt truncated for display] ...")



## Generate a named report variant

When `RUN_REPORT_GENERATION = True`, this cell will:
- generate the report with the current `REPORT_GENERATION_PROMPT`
- save it under `positive_cases/output/<variant_name>/agentic_report.md`
- copy the source memos into that variant directory
- save the exact prompt used to `report_generation_prompt.md`
- register the variant in `positive_cases/output/report_variant_registry.json`


In [ ]:

report_generation_result = None
if RUN_REPORT_GENERATION:
    client = pc_utils.make_openai_client()
    report_generation_result = pc_utils.generate_report_variant(
        client=client,
        variant_name=NEW_VARIANT_NAME,
        report_prompt=REPORT_GENERATION_PROMPT,
        source_mode=SOURCE_MODE,
        report_style=f"custom_from_{BASE_REPORT_STYLE}",
        model=REPORT_MODEL,
        overwrite=OVERWRITE_REPORT_VARIANT,
        repo_root=REPO_ROOT,
    )
    print(f"Generated variant: {report_generation_result['variant_name']}")
    print(f"Report saved to: {report_generation_result['report_path']}")
    print(f"Prompt saved to: {report_generation_result['prompt_path']}")
else:
    print("Skipping report generation. Set RUN_REPORT_GENERATION = True to create the new variant.")


In [ ]:

variant_registry = pc_utils.load_report_variant_registry(REPO_ROOT, refresh=True)
variant_df = pc_utils.report_variant_registry_dataframe(variant_registry)

display(
    variant_df[
        [
            "variant_name",
            "variant_type",
            "source_mode",
            "report_style",
            "report_path",
            "report_prompt_path",
        ]
    ].sort_values(["variant_type", "variant_name"]).reset_index(drop=True)
)



## Prediction by variant name

Prediction-time augmentation is selected only by variant name.

- `baseline_reasoning` means no report augmentation.
- Any other name in `PREDICTION_VARIANTS` must exist in the registry.


In [ ]:

for variant_name in PREDICTION_VARIANTS:
    if variant_name == "baseline_reasoning":
        print(f"{variant_name}: no report augmentation")
        continue
    if variant_name not in variant_registry:
        print(f"{variant_name}: not found in registry yet")
        continue
    entry = variant_registry[variant_name]
    print(f"{variant_name}")
    print(f"  report: {REPO_ROOT / entry['report_path']}")
    print(f"  prompt: {REPO_ROOT / entry['report_prompt_path']}")


In [ ]:

PREVIEW_VARIANT = next(v for v in PREDICTION_VARIANTS if v != "baseline_reasoning" and v in variant_registry)
preview_report_text = pc_utils.load_report_text_from_variant(PREVIEW_VARIANT, variant_registry, REPO_ROOT)

print(f"Previewing variant: {PREVIEW_VARIANT}")
preview_chars = 4000
print(preview_report_text[:preview_chars])
if len(preview_report_text) > preview_chars:
    print()
    print("... [report truncated for display] ...")


In [ ]:

SYSTEM_PROMPT = pc_utils.DEFAULT_SYSTEM_PROMPT
REPORT_INSTRUCTION = pc_utils.DEFAULT_REPORT_INSTRUCTION

prediction_scenarios = []
missing_prediction_variants = []
for variant_name in PREDICTION_VARIANTS:
    if variant_name == "baseline_reasoning":
        prediction_scenarios.append(
            pc_utils.make_scenario(
                name="baseline_reasoning",
                report_text=None,
                system_prompt=SYSTEM_PROMPT,
                notes="No report augmentation.",
            )
        )
    else:
        if variant_name not in variant_registry:
            missing_prediction_variants.append(variant_name)
            continue
        prediction_scenarios.append(
            pc_utils.scenario_from_variant_name(
                variant_name,
                variant_registry,
                repo_root=REPO_ROOT,
                system_prompt=SYSTEM_PROMPT,
                report_instruction=REPORT_INSTRUCTION,
            )
        )

if missing_prediction_variants:
    message = (
        "These prediction variants are not registered yet: "
        + ", ".join(missing_prediction_variants)
    )
    if RUN_PREDICTIONS:
        raise KeyError(message)
    print(message)

[scenario["name"] for scenario in prediction_scenarios]


In [ ]:

preview = pc_utils.preview_scenario_prompt(
    validation_df=validation_df,
    scenario=prediction_scenarios[0],
    build_user_prompt=pc_utils.build_prediction_user_prompt,
    question_index=0,
)

print("System prompt:")
print(preview["system_prompt"])
print()
print("User prompt for Q1:")
print(preview["user_prompt"])



## Run the predictions

Prediction calls are made **sequentially**, not in batch.

That is intentional in this notebook so you can:
- see progress immediately
- inspect failures or strange outputs as they happen
- stop early if the new report variant is clearly not promising


In [ ]:

client = None
pred_df = pd.DataFrame()
raw_df = pd.DataFrame()

if RUN_PREDICTIONS:
    client = pc_utils.make_openai_client()
    pred_df, raw_df = pc_utils.run_scenarios(
        validation_df=validation_df,
        scenario_specs=prediction_scenarios,
        build_user_prompt=pc_utils.build_prediction_user_prompt,
        client=client,
        model=PREDICTION_MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        pause_seconds=PAUSE_SECONDS,
        question_limit=None,
        max_retries=3,
        retry_sleep_seconds=2.0,
        verbose=True,
    )
    display(pred_df)
else:
    print("Skipping prediction calls. Set RUN_PREDICTIONS = True to run them.")


## Evaluate metrics

In [ ]:

metrics_df = pd.DataFrame()
if not pred_df.empty:
    metrics_df = pc_utils.evaluate_validation_predictions(
        pred_df=pred_df,
        validation_df=validation_df,
        n_boot=N_BOOT,
        seed=0,
    )
    display(metrics_df)
else:
    print("No predictions yet. Run the prediction cell first.")


In [ ]:

if not raw_df.empty:
    display(raw_df[["variation", "question", "prediction", "error", "reasoning"]].head(20))
else:
    print("No raw prediction rows yet.")



## Save the run

This stores predictions, metrics, raw model outputs, scenario definitions, and prompt previews under `results/notebook_positive_case_prompt_lab/`.


In [ ]:

if not pred_df.empty:
    run_dir = pc_utils.save_prompt_lab_run(
        run_name=RUN_NAME,
        scenario_specs=prediction_scenarios,
        pred_df=pred_df,
        metrics_df=metrics_df,
        raw_df=raw_df,
        validation_df=validation_df,
        build_user_prompt=pc_utils.build_prediction_user_prompt,
        repo_root=REPO_ROOT,
    )
    print(run_dir)
else:
    print("No prediction outputs to save yet.")



## Suggested workflow

1. Look through the existing registry table.
2. Open an existing variant's `report_generation_prompt.md` and `agentic_report.md`.
3. Edit only `REPORT_PROMPT_ADDENDUM`.
4. Generate a new named variant.
5. Add that variant name to `PREDICTION_VARIANTS`.
6. Run sequential predictions and compare metrics.
